*Módulo 7 de 9*

> **Prefer English?** Open [`07_machine_learning_from_zero.ipynb`](../en/07_machine_learning_from_zero.ipynb) — it is the same module, in English.


# 🤖 Módulo 7 — Aprendizaje automático desde cero

🧭 **Objetivos** — entender qué es un **modelo**, conocer el **árbol de
decisión** y el **bosque aleatorio**, aprender la regla de oro de la
evaluación honesta (**división entrenamiento/prueba**), entrenar un
clasificador con tus parcelas etiquetadas, y *leer* el reporte de calidad:
**exactitud**, **precisión/exhaustividad**, **F1**, **kappa**, y la **matriz
de confusión** — más los errores de **omisión** vs **comisión**.

📚 **¿Qué es un modelo?** Un modelo es una función aprendida de ejemplos. Le
mostramos parcelas cuyo cultivo conocemos (variables → etiqueta), y aprende
reglas para predecir el cultivo de parcelas que nunca vio. Esto es
**aprendizaje supervisado**.

📚 **Árbol de decisión → bosque aleatorio.** Un **árbol de decisión** hace
preguntas sí/no sobre las variables ("¿la media de NIR > 3000?") hasta una
hoja que nombra un cultivo. Un solo árbol se sobreajusta — memoriza rarezas.
Un **bosque aleatorio** hace crecer cientos de árboles, cada uno sobre una
rebanada aleatoria de los datos y variables, y los deja **votar**. La
multitud es mucho más exacta y estable que cualquier árbol solo.

📚 **La regla de oro.** Nunca juzgues un modelo con los datos con que se
entrenó — por supuesto que salió perfecto en esos. Divide las parcelas
etiquetadas en un **conjunto de entrenamiento** (del que aprende) y un
**conjunto de prueba** (apartado, usado solo para calificar). La exactitud
en el conjunto de *prueba* estima cómo le irá en el mapa real, no visto.

![entrenamiento](../../anim/es/08_training.svg)


## Reconstruye variables y etiquetas

Como en el Módulo 6, un kernel limpio significa que volvemos a crear las
parcelas, sus variables, y las etiquetas puras antes de entrenar.


In [ ]:
# Trae el tile del taller (pocos MB; queda en caché tras la primera descarga)
import os, sys

async def trae_archivo(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await trae_archivo("crop_tile_384.tif")
print("Tile listo:", TILE)

In [ ]:
import numpy as np, rasterio, json
import scipy.ndimage, sklearn.cluster
import shepherd_wasm

async def trae_archivo(name):
    import os, sys
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url); open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request; urllib.request.urlretrieve(url, dest)
    return dest

with rasterio.open(TILE) as src:
    img = src.read(); nombres_banda = list(src.descriptions)
LABELS  = await trae_archivo("crop_labels_384.tif")
NOMBRES = await trae_archivo("class_names.json")

resultado = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0, fixedKMeansInit=True)
seg = resultado.segimg.astype(np.int32); n_seg = int(seg.max())

plano = seg.ravel(); conteos = np.bincount(plano, minlength=n_seg+1).astype(float)
conteos[conteos == 0] = 1
variables = np.zeros((n_seg+1, len(nombres_banda)*2), dtype=np.float32)
for b in range(len(nombres_banda)):
    v = img[b].ravel().astype(np.float64)
    s1 = np.bincount(plano, weights=v, minlength=n_seg+1)
    s2 = np.bincount(plano, weights=v*v, minlength=n_seg+1)
    m = s1/conteos; var = np.maximum(s2/conteos - m*m, 0)
    variables[:, 2*b], variables[:, 2*b+1] = m, np.sqrt(var)

with rasterio.open(LABELS) as src:
    lab = src.read(1)
nombres_clase = {int(k): v for k, v in json.load(open(NOMBRES)).items()}
etiqueta_parcela = np.zeros(n_seg+1, dtype=int)
for sid in np.unique(seg[lab > 0]):
    u = np.unique(lab[(seg == sid) & (lab > 0)])
    if len(u) == 1: etiqueta_parcela[sid] = u[0]
ids_entrena = np.flatnonzero(etiqueta_parcela)
print(f"Listo: {len(ids_entrena)} parcelas etiquetadas, {variables.shape[1]} variables c/u")

## Divide, entrena, y califica con honestidad

`train_test_split` aparta el 30% de las parcelas para prueba,
**estratificado** para que cada cultivo esté en ambas partes. Ajustamos un
`RandomForestClassifier` con el 70% de entrenamiento, y lo calificamos con el
30% intacto usando un `classification_report`.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = variables[ids_entrena]
y = etiqueta_parcela[ids_entrena]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

modelo = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
modelo.fit(X_tr, y_tr)

presentes = sorted(np.unique(y_te))
print(classification_report(
    y_te, modelo.predict(X_te),
    labels=presentes, target_names=[nombres_clase[c] for c in presentes]))

## Lee el reporte

- **precision** de un cultivo = de las parcelas que el modelo *llamó* así,
  ¿cuántas lo eran de verdad? Precisión baja = error de **comisión** (falsas
  alarmas).
- **recall** de un cultivo = de las parcelas que *en verdad* eran ese
  cultivo, ¿cuántas atrapó el modelo? Recall bajo = error de **omisión**
  (se le escaparon).
- **f1-score** = el balance de precisión y recall en un solo número.
- **accuracy** (abajo) = fracción global correcta entre todos los cultivos.

Ahora visualiza la **matriz de confusión**: filas = cultivo real, columnas =
predicho. La diagonal es lo correcto; las celdas fuera de la diagonal
muestran *qué* cultivos se confunden con cuáles — mucho más informativo que
un solo número de exactitud.


In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import matplotlib.pyplot as plt

y_pred = modelo.predict(X_te)
cm = confusion_matrix(y_te, y_pred, labels=presentes)
print("Kappa de Cohen (acuerdo más allá del azar):", round(cohen_kappa_score(y_te, y_pred), 3))

fig, ax = plt.subplots(figsize=(6.5, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(presentes))); ax.set_yticks(range(len(presentes)))
ax.set_xticklabels([nombres_clase[c] for c in presentes], rotation=45, ha="right")
ax.set_yticklabels([nombres_clase[c] for c in presentes])
ax.set_xlabel("predicho"); ax.set_ylabel("real")
for i in range(len(presentes)):
    for j in range(len(presentes)):
        ax.text(j, i, cm[i, j], ha="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_title("Matriz de confusión (diagonal = correcto)")
plt.tight_layout(); plt.show()

## 🧪 Ponte a prueba

**¿Por qué la exactitud se mide en el conjunto de prueba y no en el de
entrenamiento?**

<details><summary>Ver respuesta</summary>

Un modelo puede memorizar sus datos de entrenamiento y salir casi perfecto en
ellos mientras falla en parcelas nuevas (sobreajuste). El conjunto de prueba
apartado nunca se vio durante el entrenamiento, así que su exactitud estima
con honestidad el desempeño en el mapa real, no visto.

</details>

**Un cultivo tiene precisión alta pero recall bajo. En palabras simples,
¿qué hace el modelo — y eso es omisión o comisión?**

<details><summary>Ver respuesta</summary>

Cuando dice "este cultivo" suele acertar (precisión alta), pero se le escapan
muchas parcelas verdaderas de ese cultivo (recall bajo). Perder positivos
verdaderos es error de **omisión**.

</details>

**¿Por qué preferir un bosque aleatorio sobre un solo árbol de decisión?**

<details><summary>Ver respuesta</summary>

Un árbol se sobreajusta — se aferra a rarezas de los datos de entrenamiento.
Un bosque promedia cientos de árboles variados, cancelando errores
individuales, dando predicciones más exactas y estables.

</details>


## 🔭 Profundiza

Opcional: estas tarjetas bilingües de conceptos amplían lo que acabas
de aprender (prerrequisitos, linaje a fundamentos, referencias):

- [Aprendizaje automático](https://abxda.github.io/rs-learning-audio/?id=machine-learning&lang=es)
- [Árbol de decisión](https://abxda.github.io/rs-learning-audio/?id=decision-tree&lang=es)
- [Bosque aleatorio](https://abxda.github.io/rs-learning-audio/?id=random-forest&lang=es)
- [Entrenamiento de modelos](https://abxda.github.io/rs-learning-audio/?id=model-training&lang=es)
- [Datos de entrenamiento vs prueba](https://abxda.github.io/rs-learning-audio/?id=training-dataset&lang=es)
- [Validación](https://abxda.github.io/rs-learning-audio/?id=validation&lang=es)
- [Matriz de confusión](https://abxda.github.io/rs-learning-audio/?id=confusion-matrix&lang=es)
- [Exactitud global](https://abxda.github.io/rs-learning-audio/?id=overall-accuracy&lang=es)
- [Exactitud del productor](https://abxda.github.io/rs-learning-audio/?id=producer-s-accuracy&lang=es)
- [Exhaustividad (recall)](https://abxda.github.io/rs-learning-audio/?id=recall&lang=es)
- [Puntaje F1](https://abxda.github.io/rs-learning-audio/?id=f1-score&lang=es)
- [Kappa de Cohen](https://abxda.github.io/rs-learning-audio/?id=cohen-s-kappa&lang=es)
- [Error de omisión](https://abxda.github.io/rs-learning-audio/?id=omission&lang=es)
- [Error de comisión](https://abxda.github.io/rs-learning-audio/?id=commission&lang=es)



---

[← Anterior · Módulo 6 — Las parcelas se vuelven tabla + la verdad de campo](06_variables_y_etiquetas.ipynb) · [Siguiente → · Módulo 8 — Proyecto final: el mapa de cultivos](08_proyecto_mapa_de_cultivos.ipynb)
